Imports

In [1]:
import pandas as pd
from pybaseball import statcast
import numpy as np

import warnings
warnings.filterwarnings("ignore")

Attain statcast pitch logs from each season 2020-2025

In [2]:
df_20 = statcast(start_dt="2020-07-23", end_dt="2020-10-27")

This is a large query, it may take a moment to complete


100%|██████████| 97/97 [00:33<00:00,  2.86it/s]


In [3]:
df_21 = statcast(start_dt="2021-04-21", end_dt="2021-10-03")

This is a large query, it may take a moment to complete


100%|██████████| 166/166 [00:20<00:00,  7.97it/s]


In [4]:
df_22 = statcast(start_dt="2022-04-07", end_dt="2022-10-05")

This is a large query, it may take a moment to complete


100%|██████████| 182/182 [00:23<00:00,  7.68it/s]


In [5]:
df_23 = statcast(start_dt="2023-03-30", end_dt="2023-11-01")

This is a large query, it may take a moment to complete


100%|██████████| 217/217 [01:19<00:00,  2.73it/s]


In [6]:
df_24 = statcast(start_dt="2024-03-20", end_dt="2024-10-30")

This is a large query, it may take a moment to complete


100%|██████████| 225/225 [01:22<00:00,  2.74it/s]


In [7]:
df_25 = statcast(start_dt="2025-03-18", end_dt="2025-11-01")

This is a large query, it may take a moment to complete


100%|██████████| 229/229 [00:18<00:00, 12.17it/s]


In [8]:
print("2020: ", len(df_20))
print("2021: ", len(df_21))
print("2022: ", len(df_22))
print("2023: ", len(df_23))
print("2024: ", len(df_24))
print("2025: ", len(df_25))

2020:  280398
2021:  638465
2022:  710210
2023:  732562
2024:  745349
2025:  756325


Concatenate into one df

In [9]:
df_train = pd.concat([df_20, df_21, df_22, df_23, df_24, df_25], ignore_index=True)

len(df_train)

3863309

Group pitch outcomes based on QUALITY of contact

In [10]:
print(df_train["events"].unique())
print()
print(df_train["description"].unique())

['strikeout' nan 'field_out' 'grounded_into_double_play' 'walk' 'home_run'
 'intent_walk' 'double' 'single' 'fielders_choice' 'triple' 'truncated_pa'
 'force_out' 'sac_bunt' 'hit_by_pitch' 'sac_fly' 'fielders_choice_out'
 'double_play' 'strikeout_double_play' 'field_error' 'catcher_interf'
 'sac_fly_double_play' 'triple_play' 'sac_bunt_double_play']

['called_strike' 'swinging_strike' 'ball' 'foul' 'hit_into_play'
 'blocked_ball' 'automatic_ball' 'swinging_strike_blocked' 'foul_bunt'
 'foul_tip' 'hit_by_pitch' 'missed_bunt' 'pitchout' 'bunt_foul_tip'
 'foul_pitchout' 'automatic_strike']


In [11]:
swinging_strike = ["swinging_strike", "swinging_strike_blocked", "missed_bunt"]
ball = ["ball", "blocked_ball"]
strikeout = ["strikeout", "strikeout_double_play"]
ip_out = ["grounded_into_double_play", "field_out", "double_play", "triple_play", "sac_fly", "sac_fly_double_play", "sac_bunt", "sac_bunt_double_play", "field_error", "fielders_choice_out", "fielders_choice", "force_out"]
foul = ["foul", "foul_tip", "foul_bunt", "bunt_foul_tip", "foul_pitchout"]

Create and populate 'pitch_result' column

In [12]:
df_train["pitch_result"] = np.nan

df_train.loc[df_train["description"] == "called_strike", "pitch_result"] = "called_strike"
df_train.loc[df_train["description"].isin(swinging_strike), "pitch_result"] = "swinging_strike"
df_train.loc[df_train["description"].isin(ball), "pitch_result"] = "ball"
df_train.loc[df_train["description"].isin(foul), "pitch_result"] = "foul"

df_train.loc[df_train["events"].isin(ip_out), "pitch_result"] = "ip_out"
df_train.loc[df_train["events"] == "single", "pitch_result"] = "single"
df_train.loc[df_train["events"] == "double", "pitch_result"] = "double"
df_train.loc[df_train["events"] == "triple", "pitch_result"] = "triple"
df_train.loc[df_train["events"] == "home_run", "pitch_result"] = "home_run"
df_train.loc[df_train["events"] == "walk", "pitch_result"] = "walk"
df_train.loc[df_train["events"].isin(strikeout), "pitch_result"] = "strikeout"
df_train.loc[df_train["events"] == "hit_by_pitch", "pitch_result"] = "hbp"

Group by pitch_result and see the change in run expectancy for each outcome. These will be used as our rv values going forward

In [13]:
pitch_rv = (
    df_train[df_train["pitch_result"].notna()]    
    .groupby("pitch_result")["delta_run_exp"]
    .mean()
)

rv_dict = {**pitch_rv.to_dict()}

rv_df = pd.DataFrame([rv_dict]).T.reset_index()
rv_df.columns = ["pitch_result", "rv"]

In [14]:
rv_df.sort_values("rv")

,pitch_result,rv
6,ip_out,-0.240909
8,strikeout,-0.220259
9,swinging_strike,-0.055701
1,called_strike,-0.050687
3,foul,-0.035470
0,ball,0.047838
11,walk,0.236474
4,hbp,0.369246
7,single,0.445591
2,double,0.728311


Save raw data with pitch_result as parquet

In [15]:
df_train.to_parquet("pre_2026_raw.parquet")